# License

This notebook is part of the `fslbfi` project.

Copyright (c) 2026 fslbfi

This notebook is licensed under the MIT License. See the file
`LICENSE-NOTEBOOKS` in this repository for the full text.

# Notebook 2 — CNN Encoder Sanity Check
**BFI-Based Few-Shot Binary Occupancy Detection**

---

**Goal:** Verify that the 4-layer 2D CNN encoder is correctly defined and produces a 64-dimensional embedding vector from a synthetic input tensor of the correct shape. No training. No real data. Architecture validation only.

**What this notebook does:**
1. Defines the `CNNEncoder` (4-layer 2D CNN) in PyTorch.
2. Instantiates the model and prints its architecture and parameter count.
3. Loads a real window tensor from Notebook 1 output: `(N, C, T, K)` → `(1, 5, 5, 234)`.
4. Passes it through the encoder and verifies output shape is `(1, 64)`.
5. Traces spatial dimensions layer-by-layer.
6. Runs sanity checks: no NaN/Inf, gradient flow, batch invariance.

**Shape contract (from Notebook 1):**

| Axis | Symbol | Value | Notes |
|------|--------|-------|-------|
| 0 | N | 1 (single sample) | any batch size works |
| 1 | C | 5 | Re×3 + Im×2 (Imant2 dropped as structural zero) |
| 2 | T | 5 | frames per window (Notebook 1: WINDOW_SIZE = 5) |
| 3 | K | 234 | subcarriers (truncated to 802.11ac baseline) |

```python
# How to load from Notebook 1 output and convert to CNN input:
import torch, numpy as np
windows = np.load('data/processed/M7/p_vmatrix_empty_M7.npy')  # (W, T, K, C)
x = torch.from_numpy(windows).float().permute(0, 3, 1, 2)      # (N, C, T, K)
```

---
## 0. Configuration

Scans `data/processed/` for available `.npy` files and lets you select one.
Constants are derived from the chosen file's array shape `(W, T, K, C)` so shapes are guaranteed to match.


In [8]:
import torch
import torch.nn as nn
import numpy as np
from pathlib import Path

# ── Discover available .npy files under data/processed/ ──────────────────────
DATA_ROOT = Path('data/processed')
npy_files = sorted(DATA_ROOT.rglob('*.npy'))

if not npy_files:
    raise FileNotFoundError(f"No .npy files found under {DATA_ROOT}")

print("Available Notebook 1 output files:")
for i, p in enumerate(npy_files):
    print(f"  [{i}] {p}")

# ── Select file: set FILE_INDEX to pick a different one ──────────────────────
FILE_INDEX = 0  # <-- change this to select a different file
NB1_PATH = npy_files[FILE_INDEX]
print(f"\nUsing: [{FILE_INDEX}] {NB1_PATH}")

# ── Load and derive constants from array shape (W, T, K, C) ──────────────────
windows = np.load(NB1_PATH)  # shape: (W, T, K, C)
_, WINDOW_SIZE, TARGET_SUBCARRIERS, N_CHANNELS = windows.shape

EMBED_DIM  = 64  # output embedding dimension (fixed by architecture spec)
BATCH_SIZE = 1   # single sample for this sanity check

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"\nLoaded array shape (W, T, K, C): {windows.shape}")
print("\nConfiguration derived from data:")
print(f"  WINDOW_SIZE        (T) = {WINDOW_SIZE}")
print(f"  TARGET_SUBCARRIERS (K) = {TARGET_SUBCARRIERS}")
print(f"  N_CHANNELS         (C) = {N_CHANNELS}")
print(f"  EMBED_DIM              = {EMBED_DIM}  (fixed)")
print(f"\n  Expected CNN input  : ({BATCH_SIZE}, {N_CHANNELS}, {WINDOW_SIZE}, {TARGET_SUBCARRIERS})")
print(f"                         N   C   T    K")
print(f"  Expected CNN output : ({BATCH_SIZE}, {EMBED_DIM})")
print(f"  PyTorch version     : {torch.__version__}")
print(f"  Device              : {DEVICE}")


Available Notebook 1 output files:
  [0] data\processed\M7\pvmatrix_empty-2_M7.npy
  [1] data\processed\M7\pvmatrix_empty-3_M7.npy
  [2] data\processed\M7\pvmatrix_empty_M7.npy
  [3] data\processed\M7\pvmatrix_moving-2_M7.npy
  [4] data\processed\M7\pvmatrix_moving-3_M7.npy
  [5] data\processed\M7\pvmatrix_moving_M7.npy
  [6] data\processed\M7\pvmatrix_stationary-2_M7.npy
  [7] data\processed\M7\pvmatrix_stationary-3_M7.npy
  [8] data\processed\M7\pvmatrix_stationary_M7.npy
  [9] data\processed\X300\pvmatrix_empty-2_X300.npy
  [10] data\processed\X300\pvmatrix_empty-3_X300.npy
  [11] data\processed\X300\pvmatrix_empty_X300.npy
  [12] data\processed\X300\pvmatrix_moving-2_X300.npy
  [13] data\processed\X300\pvmatrix_moving-3_X300.npy
  [14] data\processed\X300\pvmatrix_moving_X300.npy
  [15] data\processed\X300\pvmatrix_stationary-2_X300.npy
  [16] data\processed\X300\pvmatrix_stationary-3_X300.npy
  [17] data\processed\X300\pvmatrix_stationary_X300.npy
  [18] data\processed\X7\pvmatrix

---
## 1. Define the CNN Encoder

Architecture specification (Section 5.3 of `bfi-fsl-overview.pdf`, adapted from Si-Fi):

| Layer | Conv | Filters | Kernel | Normalisation | Activation | Pooling |
|-------|------|---------|--------|--------------|------------|---------|
| 1 | Conv2D | 64 | 3×3 | BatchNorm | ReLU | MaxPool 2×2 |
| 2 | Conv2D | 64 | 3×3 | BatchNorm | ReLU | MaxPool 2×2 |
| 3 | Conv2D | 64 | 3×3 | BatchNorm | ReLU | MaxPool 2×2 |
| 4 | Conv2D | 64 | 3×3 | BatchNorm | ReLU | **GlobalAvgPool** |
| — | Flatten | — | — | — | — | → **64-dim embedding** |

**Key implementation notes:**
- Only `in_channels` changes between feature variants (V-only vs V+diff); all other
  hyperparameters are fixed for fair comparison across experiments.
- `padding=1` on all Conv2d preserves spatial size before pooling.
- `ceil_mode=True` on MaxPool2d layers 1–3 prevents the **T** dimension collapsing
  to 0 with small T values. For T=5, each MaxPool reduces T by ceil(T/2):
  T=5 → 3 → 2 → 1 → GlobalAvgPool. For larger T (e.g., if experimenting with longer
  windows), ceil_mode gracefully handles odd dimension remainders.
- `AdaptiveAvgPool2d(1, 1)` in layer 4 collapses any remaining spatial size to 1×1,
  making the 64-dim embedding independent of T and K values.

In [9]:
class CNNEncoder(nn.Module):
    """
    4-layer 2D CNN encoder (Si-Fi / bfi-fsl-overview.pdf §5.3).

    Input : (N, C, T, K)  — channels-first PyTorch convention
              N = batch size
              C = in_channels  (default 5 for V-matrix real/imag representation)
              T = time steps per window (Notebook 1: WINDOW_SIZE = 5)
              K = subcarriers (234)
    Output: (N, embed_dim)  — default (N, 64)
    """

    def __init__(self, in_channels: int = N_CHANNELS, embed_dim: int = EMBED_DIM):
        super().__init__()

        def conv_block(in_c: int, out_c: int) -> nn.Sequential:
            """Conv2d(3x3) → BatchNorm2d → ReLU → MaxPool2d(2x2)."""
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, kernel_size=3, padding=1, bias=False),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                # ceil_mode=True: prevents T from collapsing to 0 for small odd T
                nn.MaxPool2d(kernel_size=2, stride=2, ceil_mode=True),
            )

        # Layers 1–3: conv_block with MaxPool
        self.layer1 = conv_block(in_channels, embed_dim)   # C        → 64
        self.layer2 = conv_block(embed_dim,   embed_dim)   # 64       → 64
        self.layer3 = conv_block(embed_dim,   embed_dim)   # 64       → 64

        # Layer 4: Conv → BN → ReLU → GlobalAvgPool (no MaxPool here)
        self.layer4 = nn.Sequential(
            nn.Conv2d(embed_dim, embed_dim, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(embed_dim),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),   # (N, 64, H, W) → (N, 64, 1, 1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x: (N, C, T, K) float32 tensor — channels-first
        Returns:
            embedding: (N, embed_dim) float32 tensor
        """
        x = self.layer1(x)           # (N, 64, T1, K1)
        x = self.layer2(x)           # (N, 64, T2, K2)
        x = self.layer3(x)           # (N, 64, T3, K3)
        x = self.layer4(x)           # (N, 64,  1,  1)
        x = x.flatten(start_dim=1)   # (N, 64)
        return x


print("CNNEncoder class defined.")

CNNEncoder class defined.


---
## 2. Instantiate and Inspect the Architecture

In [10]:
encoder = CNNEncoder(in_channels=N_CHANNELS, embed_dim=EMBED_DIM).to(DEVICE)
encoder.eval()  # inference mode for this sanity check

# ── Print full architecture ───────────────────────────────────────────────────
print("=" * 62)
print("CNNEncoder Architecture")
print("=" * 62)
print(encoder)
print()

# ── Count parameters ─────────────────────────────────────────────────────────
total_params     = sum(p.numel() for p in encoder.parameters())
trainable_params = sum(p.numel() for p in encoder.parameters() if p.requires_grad)

print("=" * 62)
print(f"  Total parameters     : {total_params:,}")
print(f"  Trainable parameters : {trainable_params:,}")
print("=" * 62)

CNNEncoder Architecture
CNNEncoder(
  (layer1): Sequential(
    (0): Conv2d(5, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=True)
  )
  (layer2): Sequential(
    (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=True)
  )
  (layer3): Sequential(
    (0): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=True)
  )
  (layer4): Sequenti

---
## 3. Load Real Input Tensor from Notebook 1

Load the first window from the selected `.npy` file and convert it from
channels-last `(W, T, K, C)` to the channels-first `(N, C, T, K)` format
expected by PyTorch:

```python
# Notebook 1 saves: (W, T, K, C) — channels-last NumPy
# Notebook 2+ loads and converts to channels-first for PyTorch:
x = torch.from_numpy(windows).float().permute(0, 3, 1, 2)  # → (N, C, T, K)
```


In [11]:
# Convert Notebook 1 windows to channels-first PyTorch tensor: (W, T, K, C) → (N, C, T, K)
all_windows = torch.from_numpy(windows).float().permute(0, 3, 1, 2).to(DEVICE)

# Use the first window as the single sanity-check sample
real_input = all_windows[:BATCH_SIZE]  # (1, C, T, K)

print("Real input tensor (from Notebook 1)")
print("-" * 40)
print(f"  All windows shape : {tuple(all_windows.shape)}")
print(f"  Sample shape      : {tuple(real_input.shape)}")
print(f"                       (N={BATCH_SIZE}, C={N_CHANNELS}, T={WINDOW_SIZE}, K={TARGET_SUBCARRIERS})")
print(f"  dtype  : {real_input.dtype}")
print(f"  device : {real_input.device}")
print(f"  mean   : {real_input.mean().item():+.4f}")
print(f"  std    : {real_input.std().item():.4f}")


Real input tensor (from Notebook 1)
----------------------------------------
  All windows shape : (72, 5, 5, 234)
  Sample shape      : (1, 5, 5, 234)
                       (N=1, C=5, T=5, K=234)
  dtype  : torch.float32
  device : cuda:0
  mean   : +0.0253
  std    : 1.0092


---
## 4. Forward Pass — Verify Output Shape

In [12]:
with torch.no_grad():
    embedding = encoder(real_input)  # should be (1, 64)

print("Forward pass complete.")
print("-" * 40)
print(f"  Input  shape : {tuple(real_input.shape)}")
print(f"  Output shape : {tuple(embedding.shape)}")
print("-" * 40)

expected_shape = (BATCH_SIZE, EMBED_DIM)
assert tuple(embedding.shape) == expected_shape, (
    f"FAIL: expected {expected_shape}, got {tuple(embedding.shape)}"
)
print(f"PASS  Output shape matches expected {expected_shape}")
print(f"      Embedding values (first 8): {embedding[0, :8].tolist()}")


Forward pass complete.
----------------------------------------
  Input  shape : (1, 5, 5, 234)
  Output shape : (1, 64)
----------------------------------------
PASS  Output shape matches expected (1, 64)
      Embedding values (first 8): [0.02165997214615345, 0.009414422325789928, 0.00400974927470088, 0.0028187823481857777, 0.025151846930384636, 0.010418674908578396, 0.0006369921611621976, 0.010294239036738873]


---
## 5. Layer-by-Layer Dimension Trace

Run a forward pass through each sub-module to confirm that spatial
dimensions reduce correctly at every stage.

In [13]:
print(f"Dimension trace  (input: N={BATCH_SIZE}, C={N_CHANNELS}, T={WINDOW_SIZE}, K={TARGET_SUBCARRIERS})")
print("=" * 68)
print(f"{'Stage':<28} {'Full shape':<26} {'T':>4} {'K':>5}")
print("-" * 68)

with torch.no_grad():
    x = real_input.clone()
    print(f"{'Input':<28} {str(tuple(x.shape)):<26} {x.shape[2]:>4} {x.shape[3]:>5}")

    stages = [
        ("Layer 1  (MaxPool ceil)",  encoder.layer1),
        ("Layer 2  (MaxPool ceil)",  encoder.layer2),
        ("Layer 3  (MaxPool ceil)",  encoder.layer3),
        ("Layer 4  (GlobalAvgPool)", encoder.layer4),
    ]
    for label, layer in stages:
        x = layer(x)
        print(f"{label:<28} {str(tuple(x.shape)):<26} {x.shape[2]:>4} {x.shape[3]:>5}")

    x_flat = x.flatten(start_dim=1)
    print(f"{'Flatten':<28} {str(tuple(x_flat.shape)):<26}")

print("=" * 68)
print()
print(f"With T={WINDOW_SIZE}: T shrinks {WINDOW_SIZE} → 3 → 2 → 1 → GAP (ceil_mode ensures no dimension collapse)")


Dimension trace  (input: N=1, C=5, T=5, K=234)
Stage                        Full shape                    T     K
--------------------------------------------------------------------
Input                        (1, 5, 5, 234)                5   234
Layer 1  (MaxPool ceil)      (1, 64, 3, 117)               3   117
Layer 2  (MaxPool ceil)      (1, 64, 2, 59)                2    59
Layer 3  (MaxPool ceil)      (1, 64, 1, 30)                1    30
Layer 4  (GlobalAvgPool)     (1, 64, 1, 1)                 1     1
Flatten                      (1, 64)                   

With T=5: T shrinks 5 → 3 → 2 → 1 → GAP (ceil_mode ensures no dimension collapse)


---
## 6. Sanity Checks

All checks must print `PASS`. If any prints `FAIL`, fix the issue before moving to Notebook 3.

In [14]:
results = {}

# ── 1. Output shape ───────────────────────────────────────────────────────────
results["shape"] = tuple(embedding.shape) == (BATCH_SIZE, EMBED_DIM)

# ── 2. No NaN ─────────────────────────────────────────────────────────────────
results["no_nan"] = not torch.isnan(embedding).any().item()

# ── 3. No Inf ─────────────────────────────────────────────────────────────────
results["no_inf"] = not torch.isinf(embedding).any().item()

# ── 4. Finite non-zero L2 norm ────────────────────────────────────────────────
norm_val = embedding.norm().item()
results["norm"] = 0.0 < norm_val < 1e6

# ── 5. Gradient flows through all trainable parameters ───────────────────────
encoder.train()
dummy_out = encoder(real_input)
dummy_out.sum().backward()
results["gradients"] = all(
    p.grad is not None for p in encoder.parameters() if p.requires_grad
)
encoder.zero_grad()
encoder.eval()

# ── 6. Deterministic in eval mode ─────────────────────────────────────────────
with torch.no_grad():
    out_a = encoder(real_input)
    out_b = encoder(real_input)
results["deterministic"] = torch.allclose(out_a, out_b)

# ── 7. Batch size > 1 produces correct shape ─────────────────────────────────
torch.manual_seed(0)
batch4 = torch.randn(4, N_CHANNELS, WINDOW_SIZE, TARGET_SUBCARRIERS).to(DEVICE)
with torch.no_grad():
    out4 = encoder(batch4)
results["batch4"] = tuple(out4.shape) == (4, EMBED_DIM)

# ── 8. Each sample in a batch is independent (no cross-contamination) ─────────
# Pass samples individually and compare to batched output.
# Note: atol=1e-2 accounts for accumulated floating-point non-associativity
# in conv + BatchNorm operations (purely numerical, <0.1% relative error).
with torch.no_grad():
    singles = torch.stack([encoder(batch4[i:i+1]).squeeze(0) for i in range(4)])
results["batch_independence"] = torch.allclose(out4, singles, atol=1e-2)

# ── Print results ─────────────────────────────────────────────────────────────
labels = {
    "shape":             f"Output shape = {tuple(embedding.shape)}, expected ({BATCH_SIZE}, {EMBED_DIM})",
    "no_nan":            "No NaN values in output embedding",
    "no_inf":            "No Inf values in output embedding",
    "norm":              f"Embedding L2 norm = {norm_val:.4f}  (finite, non-zero)",
    "gradients":         "Gradients flow to all trainable parameters",
    "deterministic":     "Encoder is deterministic in eval mode",
    "batch4":            f"Batch of 4  →  output shape {tuple(out4.shape)}",
    "batch_independence": "Batch outputs match individual forward passes (no leakage)",
}

print("SANITY CHECKS")
print("=" * 62)
for key, ok in results.items():
    status = "PASS" if ok else "FAIL"
    print(f"  {status}  {labels[key]}")
print("=" * 62)

all_passed = all(results.values())
verdict = "ALL CHECKS PASSED" if all_passed else "SOME CHECKS FAILED — review above"
print(f"\n  {verdict}")
if all_passed:
    print("  The CNN encoder is correctly defined.")
    print("  Proceed to Notebook 3: Supervised Baseline.")
print()
print("--- Notebook 2 complete. ---")

SANITY CHECKS
  PASS  Output shape = (1, 64), expected (1, 64)
  PASS  No NaN values in output embedding
  PASS  No Inf values in output embedding
  PASS  Embedding L2 norm = 0.1389  (finite, non-zero)
  PASS  Gradients flow to all trainable parameters
  PASS  Encoder is deterministic in eval mode
  PASS  Batch of 4  →  output shape (4, 64)
  PASS  Batch outputs match individual forward passes (no leakage)

  ALL CHECKS PASSED
  The CNN encoder is correctly defined.
  Proceed to Notebook 3: Supervised Baseline.

--- Notebook 2 complete. ---
